<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
"""
I suspect pages with declining trend_pct are ranking worse overall.
both in their typical per-query position (avg_position) and in their total position burden across all their queries (sum_position).
"""

'\nI suspect pages with declining trend_pct are ranking worse overall.\nboth in their typical per-query position (avg_position) and in their total position burden across all their queries (sum_position).\n'

In [2]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [3]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [6]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [7]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [8]:
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,trend_pct
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,-47.280702
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,263.157895
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,-34.539322
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,4.697987
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,-2.869757


In [9]:
negative_mean = df.loc[df['trend_pct'] < 0, 'trend_pct'].mean()
positive_mean = df.loc[df['trend_pct'] > 0, 'trend_pct'].mean()
conditions = [
    df['trend_pct'] < negative_mean,
    (df['trend_pct'] < 0) & (df['trend_pct'] >= negative_mean),
    (df['trend_pct'] >= 0) & (df['trend_pct'] < positive_mean),   # now includes 0
    df['trend_pct'] >= positive_mean
]
ranks = ['Sharp decline', 'Mild decline', 'Mild growth', 'Strong growth']
df['trend_dir'] = np.select(conditions, ranks, default=None)

In [10]:
display(df.groupby('trend_dir').agg(
    {
        'gsc_sum_position': ['mean', 'count'],
        'gsc_avg_position': ['mean', 'count'],
    }
))

gsc_sum_position          gsc_avg_position         
                          mean    count             mean    count
trend_dir                                                        
Mild decline        996.146252  1257661        14.197085  1190965
Mild growth         501.813023   560059        12.195070   521437
Sharp decline      1242.927716  1183133        19.375020  1055130
Strong growth       228.144671   257487        14.198881   219478

In [11]:
"""
gsc_avg_position averages across a page's queries,
but since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),
a page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.
This likely explains some of the scrambled middle-bucket ordering — sum_position,
while confounded by query volume, doesn't suffer from this specific averaging distortion.
"""

"\ngsc_avg_position averages across a page's queries, \nbut since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers), \na page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries. \nThis likely explains some of the scrambled middle-bucket ordering — sum_position, \nwhile confounded by query volume, doesn't suffer from this specific averaging distortion.\n"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
position_share=df['gsc_sum_position']/df['gsc_sum_position'].sum()
score=-df['trend_pct']*position_share
df['score']=score
df['score'] = df['score'] * 10000

In [13]:
df = df[['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score']].copy()

In [14]:
conditions = [
    (df['trend_pct'] < negative_mean) & (position_share > position_share.median()),
    (df['trend_pct'] < negative_mean) & (position_share <= position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0) & (position_share > position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0),
    (df['trend_pct'] >= 0) & (position_share > position_share.median()),
]
codes = [
    'strong declining trend with low page position',
    'strong declining trend',
    'mild declining trend with low page position',
    'mild declining trend',
    'low page position',
]
df['reason_code'] = np.select(conditions, codes, default='STABLE')


In [15]:
df['action'] = np.select(
    [
        df['reason_code'] == 'strong declining trend with low page position',
        df['reason_code'].isin(['strong declining trend', 'mild declining trend with low page position']),
    ],
    ['REFRESH', 'MONITOR'],
    default='SKIP'
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
import os
os.makedirs('work/outputs', exist_ok=True)

output_cols = ['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score', 'reason_code', 'action']
ranked_queue = df.sort_values(by='score', ascending=False)[output_cols]
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

In [23]:
df.shape

(9841378, 7)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.